# DrugCentral Remapping Notebook
Updates the MIND dataset with the latest DrugCentral FDA-approved indications.

**Before running:**
1. Add your MIND dataset as input data (Add Data button)
2. Run all cells top to bottom
3. Download `mind_updated.tsv` and `mapping_report.csv` from Output


In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────
import subprocess, sys
for pkg in ['rapidfuzz','tqdm']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'],check=True)
print('Dependencies installed')


In [ ]:
# ── CELL 2: Find MIND dataset ──────────────────────────────────────────────
import os
from pathlib import Path

# Show all input files so we can find MIND
print('Files in /kaggle/input:')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = Path(root)/f
        size  = fpath.stat().st_size / 1024**2
        print(f'  {fpath}  ({size:.1f} MB)')


In [ ]:
# ── CELL 3: Set MIND path ─────────────────────────────────────────────────
# After running Cell 2, look at the output and find your mind.tsv file
# Copy its full path and paste below

MIND_PATH = ''  # <-- paste path here e.g. '/kaggle/input/mind-dataset/mind.tsv'

# Auto-detect if not set
if not MIND_PATH:
    candidates = sorted(Path('/kaggle/input').glob('**/*.tsv'),
                        key=lambda f: f.stat().st_size, reverse=True)
    if candidates:
        MIND_PATH = str(candidates[0])
        print(f'Auto-detected MIND: {MIND_PATH}')
    else:
        print('ERROR: No TSV found. Please add MIND dataset via Add Data button')

OUTPUT_PATH = '/kaggle/working/mind_updated.tsv'
REPORT_PATH = '/kaggle/working/mapping_report.csv'
print(f'MIND path:   {MIND_PATH}')
print(f'Output path: {OUTPUT_PATH}')


In [ ]:
# ── CELL 4: Load and inspect MIND ─────────────────────────────────────────
import pandas as pd
import numpy as np
import re
from collections import defaultdict

print('Loading MIND...')
mrn = pd.read_csv(MIND_PATH, sep='\t', header=None,
                  names=['head','relation','tail'])
print(f'Total triples: {len(mrn):,}')
print(f'Relations ({mrn.relation.nunique()}):')
print(mrn.relation.value_counts().to_string())

nodes = pd.Series(pd.unique(mrn[['head','tail']].values.ravel())).dropna().unique()
print(f'\nUnique nodes: {len(nodes):,}')

# Show existing indication edges
ind = mrn[mrn.relation == 'indication']
print(f'\nExisting indication edges: {len(ind):,}')
print('Sample:')
print(ind.head(5).to_string())


In [ ]:
# ── CELL 5: Build lookup indexes from MRN nodes ───────────────────────────
from tqdm import tqdm

def norm(s):
    if pd.isna(s): return ''
    return re.sub(r'[^a-z0-9 ]', '', str(s).lower().strip())

def get_cui(s):
    if pd.isna(s): return None
    m = re.search(r'C\d{7}', str(s))
    return m.group(0) if m else None

def get_mesh(s):
    if pd.isna(s): return None
    m = re.search(r'D\d{6}', str(s))
    return m.group(0) if m else None

print('Building indexes...')
name_idx, cui_idx, mesh_idx = {}, {}, {}

for node in tqdm(nodes):
    n = norm(node)
    if n: name_idx[n] = node
    c = get_cui(node)
    if c: cui_idx[c] = node
    m = get_mesh(node)
    if m: mesh_idx[m] = node

all_names = list(name_idx.keys())
print(f'By name:     {len(name_idx):,}')
print(f'By UMLS CUI: {len(cui_idx):,}')
print(f'By MeSH ID:  {len(mesh_idx):,}')


In [ ]:
# ── CELL 6: Download DrugCentral ───────────────────────────────────────────
import requests, io

URLS = [
    'https://unmtid-shinyapps.net/download/drugcentral/drug_indications.tsv',
    'https://drugcentral.org/static/download/drug_indications.tsv',
]

dc = None
for url in URLS:
    try:
        print(f'Downloading from {url}...')
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        dc = pd.read_csv(io.StringIO(r.text), sep='\t', low_memory=False)
        print(f'Downloaded {len(dc):,} rows')
        print(f'Columns: {list(dc.columns)}')
        break
    except Exception as e:
        print(f'Failed: {e}')

if dc is None:
    print('ERROR: Could not download DrugCentral')
    print('Go to https://drugcentral.org/download, download drug_indications.tsv')
    print('Upload it as a Kaggle dataset and load it manually')
else:
    print('\nSample rows:')
    print(dc.head(3).to_string())


In [ ]:
# ── CELL 7: Match DrugCentral to MRN ──────────────────────────────────────
try:
    from rapidfuzz import fuzz, process as rfprocess
    FUZZY = True
    print('Fuzzy matching enabled')
except:
    FUZZY = False
    print('Fuzzy matching disabled')

def match_entity(name, cui, mesh, threshold=88):
    # 1. UMLS CUI
    if cui and not pd.isna(cui):
        c = get_cui(str(cui))
        if c and c in cui_idx: return cui_idx[c], 'umls_cui'
    # 2. MeSH
    if mesh and not pd.isna(mesh):
        m = get_mesh(str(mesh))
        if m and m in mesh_idx: return mesh_idx[m], 'mesh'
    # 3. Exact name
    n = norm(name)
    if n and n in name_idx: return name_idx[n], 'exact_name'
    # 4. Fuzzy
    if FUZZY and n and all_names:
        res = rfprocess.extractOne(n, all_names,
                                   scorer=fuzz.token_sort_ratio,
                                   score_cutoff=threshold)
        if res:
            matched, score, _ = res
            return name_idx[matched], f'fuzzy_{score}'
    return None, None

# Detect column names
drug_col    = next((c for c in dc.columns if 'drug' in c.lower() and 'name' in c.lower()), dc.columns[0])
disease_col = next((c for c in dc.columns if any(x in c.lower() for x in ['concept','indication','disease','snomed_full'])), None)
cui_col     = next((c for c in dc.columns if 'umls' in c.lower() and 'cui' in c.lower()), None)
mesh_col    = next((c for c in dc.columns if 'mesh' in c.lower()), None)
status_col  = next((c for c in dc.columns if 'status' in c.lower()), None)

print(f'Drug col:    {drug_col}')
print(f'Disease col: {disease_col}')
print(f'CUI col:     {cui_col}')
print(f'MeSH col:    {mesh_col}')
print(f'Status col:  {status_col}')

# Filter approved
if status_col:
    approved = dc[dc[status_col].str.upper().isin(['APPROVED','FDA','EMA'])].copy()
    print(f'\nApproved: {len(approved):,} of {len(dc):,}')
else:
    approved = dc.copy()
    print(f'No status col — using all {len(approved):,} rows')

# Match
print('\nMatching indications to MRN nodes...')
rows = []
drug_stats, dis_stats = defaultdict(int), defaultdict(int)

for _, row in tqdm(approved.iterrows(), total=len(approved)):
    drug_name = row.get(drug_col, '')
    dis_name  = row.get(disease_col, '') if disease_col else ''
    cui       = row.get(cui_col,  None) if cui_col  else None
    mesh      = row.get(mesh_col, None) if mesh_col else None

    drug_node, drug_m = match_entity(drug_name, None, None)
    dis_node,  dis_m  = match_entity(dis_name, cui, mesh)

    drug_stats[drug_m or 'unmatched'] += 1
    dis_stats[dis_m  or 'unmatched']  += 1

    if drug_node and dis_node:
        rows.append({
            'head': drug_node, 'relation': 'indication', 'tail': dis_node,
            'dc_drug': drug_name, 'dc_disease': dis_name,
            'drug_match': drug_m, 'dis_match': dis_m,
        })

matched = pd.DataFrame(rows).drop_duplicates(subset=['head','relation','tail'])
print(f'\nMatched: {len(matched):,} unique indication triples')
print(f'Drug methods:    {dict(drug_stats)}')
print(f'Disease methods: {dict(dis_stats)}')


In [ ]:
# ── CELL 8: Update MIND and save ───────────────────────────────────────────
existing     = mrn[mrn.relation == 'indication']
existing_set = set(zip(existing.head, existing.tail))

new_triples           = matched[['head','relation','tail']].copy()
new_triples['is_new'] = new_triples.apply(
    lambda r: (r.head, r.tail) not in existing_set, axis=1)
genuinely_new = new_triples[new_triples.is_new].drop('is_new', axis=1)

print(f'Existing indication edges:  {len(existing):,}')
print(f'Newly matched:              {len(matched):,}')
print(f'Genuinely new (not in MRN): {len(genuinely_new):,}')
print(f'Already in MRN:             {len(matched)-len(genuinely_new):,}')

updated = pd.concat([mrn, genuinely_new], ignore_index=True)
print(f'\nUpdated MRN: {len(updated):,} triples (was {len(mrn):,})')

# Save
updated[['head','relation','tail']].to_csv(
    OUTPUT_PATH, sep='\t', index=False, header=False)
matched.to_csv(REPORT_PATH, index=False)

print(f'\nSaved: {OUTPUT_PATH}')
print(f'Saved: {REPORT_PATH}')

# Manuscript numbers
total = len(existing) + len(genuinely_new)
print(f'\n{"="*50}')
print(f'MANUSCRIPT NUMBERS:')
print(f'  Original (2021):   5,558')
print(f'  Current MIND:      {len(existing):,}')
print(f'  New from mapping:  {len(genuinely_new):,}')
print(f'  Total updated:     {total:,}')
print(f'  Improvement:       +{len(genuinely_new):,} ({len(genuinely_new)/max(len(existing),1)*100:.1f}%)')
print(f'{"="*50}')


In [ ]:
# ── CELL 9: Preview new indications ───────────────────────────────────────
print('Sample new indication edges found:')
print(genuinely_new.head(20).to_string())

print('\nSample matched report:')
print(matched[matched.apply(
    lambda r: (r.head,r.tail) not in existing_set, axis=1
)].head(10)[['dc_drug','dc_disease','drug_match','dis_match']].to_string())
